In [28]:
import pandas as pd
import numpy as np
import re
import json

with open("../data/raw/track1_chargebacks.json") as f:
    cb_raw = json.load(f)

cb = pd.json_normalize(cb_raw)
print(cb.shape)
cb.isnull().sum() #no null values

(2884, 13)


complaint_id               0
txn_id                     0
user_id                    0
merchant_id                0
transaction_timestamp      0
reported_timestamp         0
disputed_amount            0
reason_code                0
complaint_text             0
resolution_status          0
bank_response_timestamp    0
severity                   0
channel                    0
dtype: int64

In [29]:
# complaint_id duplicates (84) == full-row duplicates (84), so these are
# true copies, not separate complaints — safe to drop outright, unlike
before = len(cb)
cb = cb.drop_duplicates()
print(f"Dropped {before - len(cb)} exact duplicate rows")

Dropped 84 exact duplicate rows


In [30]:
# Column txn_id

# 1. Standardize to match transactions' fixed format: 'TXN' + 8 zero-padded
#    digits. Raw values are inconsistent ('TXN65742' unpadded, 'txn-00003616'
#    lowercase+hyphenated) and won't match transactions' txn_id without this.
def std_txn_id(value):
    if pd.isnull(value):
        return None
    digits = re.sub(r'[^0-9]', '', str(value))
    return f"TXN{digits.zfill(8)}" if digits else None

cb['txn_id_clean'] = cb['txn_id'].apply(std_txn_id)

# FIX: check txn_id_clean, not the raw txn_id — 81 rows have txn_id as an
# empty string '' (not actual NaN), so raw.isna() misses them even though
# txn_id_clean correctly resolves to None for those rows.
cb['txn_id_status'] = np.where(cb['txn_id_clean'].isna(), 'MISSING', 'VALID')
cb['txn_id_status'].value_counts()

txn_id_status
VALID      2723
MISSING      77
Name: count, dtype: int64

In [31]:
# Columns user_id, merchant_id

# 1. Standardize both the same way: strip everything except digits,
#    re-add the canonical prefix (handles 'usr38163' vs 'USR48339' etc.)
def std_id(value, prefix):
    if pd.isnull(value):
        return None
    digits = re.sub(r'[^0-9]', '', str(value))
    return f"{prefix}{digits}" if digits else None

cb['user_id_clean'] = cb['user_id'].apply(lambda x: std_id(x, 'USR'))
cb['merchant_id_clean'] = cb['merchant_id'].apply(lambda x: std_id(x, 'MCH'))

In [32]:
# Columns transaction_timestamp, reported_timestamp, bank_response_timestamp

# 1. Enhanced flexible parser that also detects the "Midnight Trap" (date only, no time)
def parse_flex_date_with_status(value):
    # Catches actual programmatic nulls
    if pd.isnull(value):
        return pd.NaT, 'MISSING'
    
    s = str(value).strip()
    
    # Intercept text-based empty values
    if s.lower() in ['nan', 'none', 'nat', 'null', '']:
        return pd.NaT, 'MISSING'

    is_unix = s.isdigit() and len(s) == 10
    has_colon = ':' in s

    # Parse Unix epoch seconds or flexible string formats
    if is_unix:
        dt = pd.to_datetime(int(s), unit='s', errors='coerce')
    else:
        dt = pd.to_datetime(s, format='mixed', errors='coerce')

    # If parsing failed completely
    if pd.isna(dt):
        return pd.NaT, 'UNPARSEABLE/NULL'
    
    # Check if it successfully parsed but only provided a date (no time component or colon)
    if not is_unix and not has_colon:
        return dt, 'DATE_ONLY'
    
    return dt, 'VALID'

# 2. Iterate through all three timestamp columns to clean and categorize
for col in ['transaction_timestamp', 'reported_timestamp', 'bank_response_timestamp']:
    # Apply the parser and unpack the results into clean and status columns
    parsed_results = cb[col].apply(parse_flex_date_with_status)
    
    cb[f'{col}_clean'] = [res[0] for res in parsed_results]
    cb[f'{col}_status'] = [res[1] for res in parsed_results]

# 3. Check the breakdown of statuses across all timestamp fields
cb[['transaction_timestamp_status', 'reported_timestamp_status', 'bank_response_timestamp_status']].apply(
    lambda c: c.value_counts(dropna=False)
)

,transaction_timestamp_status,reported_timestamp_status,bank_response_timestamp_status
DATE_ONLY,1689,1693,1387
VALID,881,904,712
MISSING,230,203,701


In [33]:
# Column disputed_amount

# 1. Strip currency symbols/words/commas. Keep negative values VISIBLE
#    (flagged, not deleted) — 231 found, and a negative dispute amount
#    is itself worth investigating, not noise to hide.
def clean_amount(value):
    if pd.isnull(value):
        return None
    v = re.sub(r'(?i)(inr|rs\.?|₹|,|\s)', '', str(value))
    try:
        return float(v)
    except ValueError:
        return None

cb['disputed_amount_clean'] = cb['disputed_amount'].apply(clean_amount)

is_missing = cb['disputed_amount'].isna()
is_unparse = cb['disputed_amount_clean'].isna() & ~is_missing
is_negative = cb['disputed_amount_clean'] < 0
cb['disputed_amount_status'] = np.select(
    [is_negative, is_unparse, is_missing], ['NEGATIVE', 'UNPARSEABLE/NULL', 'MISSING'], default='VALID'
)
cb['disputed_amount_status'].value_counts()

disputed_amount_status
VALID               2401
NEGATIVE             220
UNPARSEABLE/NULL     179
Name: count, dtype: int64

In [34]:
cb['reason_code'].unique()

array(['Merchant Not Delivered', 'login compromised', 'customer issue',
       'no service', 'merchant service issue', 'service failed',
       'extra amount deducted', 'Service Not Provided', 'dispute raised',
       'UNAUTHORISED', 'double debit', 'Fraud Suspected', 'fraud',
       'charged twice', 'Account Takeover', 'Wrong Amount',
       'Unauthorized Transaction', 'unauthorized_transaction',
       'DUP_DEBIT', 'incorrect amount', 'account hacked', 'scam',
       'unauth txn', 'Duplicate Debit', 'delivery issue', 'FRAUD',
       'Customer Dispute', 'suspicious transaction', 'ATO',
       'not delivered', 'item not received', 'complaint',
       'not done by me', 'amount mismatch'], dtype=object)

In [35]:
# Column reason_code

# ⚠️ Judgment call, not a verified fact: raw data has 34 variants that
# read like they cluster into 5 real dispute categories (e.g. 'ATO',
# 'account hacked', 'not done by me', and 'FRAUD' all describe the same
# underlying complaint type). This mapping is MY interpretation of that
# clustering — worth a quick sanity check on your end before trusting it.
reason_map = {
    'account takeover': 'UNAUTHORIZED_TRANSACTION', 'ato': 'UNAUTHORIZED_TRANSACTION',
    'account hacked': 'UNAUTHORIZED_TRANSACTION', 'login compromised': 'UNAUTHORIZED_TRANSACTION',
    'not done by me': 'UNAUTHORIZED_TRANSACTION', 'unauth txn': 'UNAUTHORIZED_TRANSACTION',
    'unauthorised': 'UNAUTHORIZED_TRANSACTION', 'unauthorized transaction': 'UNAUTHORIZED_TRANSACTION',
    'unauthorized_transaction': 'UNAUTHORIZED_TRANSACTION', 'suspicious transaction': 'UNAUTHORIZED_TRANSACTION',
    'fraud': 'UNAUTHORIZED_TRANSACTION', 'fraud suspected': 'UNAUTHORIZED_TRANSACTION', 'scam': 'UNAUTHORIZED_TRANSACTION',

    'duplicate debit': 'DUPLICATE_DEBIT', 'dup_debit': 'DUPLICATE_DEBIT',
    'charged twice': 'DUPLICATE_DEBIT', 'double debit': 'DUPLICATE_DEBIT',

    'wrong amount': 'WRONG_AMOUNT', 'incorrect amount': 'WRONG_AMOUNT',
    'amount mismatch': 'WRONG_AMOUNT', 'extra amount deducted': 'WRONG_AMOUNT',

    'no service': 'SERVICE_NOT_PROVIDED', 'not delivered': 'SERVICE_NOT_PROVIDED',
    'item not received': 'SERVICE_NOT_PROVIDED', 'delivery issue': 'SERVICE_NOT_PROVIDED',
    'merchant not delivered': 'SERVICE_NOT_PROVIDED', 'service not provided': 'SERVICE_NOT_PROVIDED',
    'merchant service issue': 'SERVICE_NOT_PROVIDED', 'service failed': 'SERVICE_NOT_PROVIDED',

    'customer dispute': 'GENERAL_COMPLAINT', 'complaint': 'GENERAL_COMPLAINT',
    'dispute raised': 'GENERAL_COMPLAINT', 'customer issue': 'GENERAL_COMPLAINT',
}
cb['reason_code_clean'] = cb['reason_code'].astype(str).str.strip().str.lower().map(reason_map).fillna('OTHER')
cb['reason_code_clean'].value_counts()

reason_code_clean
UNAUTHORIZED_TRANSACTION    1042
SERVICE_NOT_PROVIDED         704
GENERAL_COMPLAINT            376
DUPLICATE_DEBIT              352
WRONG_AMOUNT                 326
Name: count, dtype: int64

In [36]:
# Column complaint_text

# 1. Free-text narrative field — normalize whitespace/casing only.
#    Not restructured into categories since it reads as open commentary
#    (reason_code above is the actual categorical field).
cb['complaint_text_clean'] = cb['complaint_text'].astype(str).str.strip().str.capitalize()

In [37]:
# Column resolution_status

# 1. 13 raw variants (including 'WIP' as an alias for in-progress) -> 6 canonical buckets
resolution_map = {
    'open': 'OPEN',
    'in progress': 'IN_PROGRESS', 'in_progress': 'IN_PROGRESS', 'wip': 'IN_PROGRESS',
    'closed': 'CLOSED',
    'resolved': 'RESOLVED',
    'rejected': 'REJECTED',
    'pending bank': 'PENDING_BANK', 'pending_bank': 'PENDING_BANK',
}
cb['resolution_status_clean'] = cb['resolution_status'].astype(str).str.strip().str.lower().map(resolution_map).fillna('UNKNOWN')
cb['resolution_status_clean'].value_counts()

resolution_status_clean
IN_PROGRESS     599
PENDING_BANK    458
REJECTED        443
CLOSED          442
OPEN            435
RESOLVED        423
Name: count, dtype: int64

In [38]:
cb['severity'].unique()

array(['Critical', 'H', 'P4', 'Low', 'LOW', 'M', 'P2', 'Medium', 'P3',
       'CRITICAL', 'MEDIUM', 'P1', 'HIGH', 'High', 'L', 'CRIT'],
      dtype=object)

In [39]:
# Column severity

# ⚠️ Assumption, not confirmed by the data: raw values mix words, single
# letters, AND a P1-P4 priority scale. Assuming P1/Critical = most severe,
# P4/Low = least severe (standard convention, but worth double-checking).
severity_map = {
    'critical': 'CRITICAL', 'crit': 'CRITICAL', 'p1': 'CRITICAL',
    'high': 'HIGH', 'h': 'HIGH', 'p2': 'HIGH',
    'medium': 'MEDIUM', 'm': 'MEDIUM', 'p3': 'MEDIUM',
    'low': 'LOW', 'l': 'LOW', 'p4': 'LOW',
}
cb['severity_clean'] = cb['severity'].astype(str).str.strip().str.lower().map(severity_map).fillna('UNKNOWN')
cb['severity_clean'].value_counts()

severity_clean
MEDIUM      1024
LOW          959
HIGH         604
CRITICAL     213
Name: count, dtype: int64

In [40]:
# Column channel

# 1. Casing-only inconsistency (ivr/IVR, chatbot/CHATBOT) -> fixed set of 5
channel_map = {
    'email': 'EMAIL', 'ivr': 'IVR', 'call center': 'CALL_CENTER',
    'chatbot': 'CHATBOT', 'branch': 'BRANCH', 'app': 'APP',
}
cb['channel_clean'] = cb['channel'].astype(str).str.strip().str.lower().map(channel_map).fillna('UNKNOWN')
cb['channel_clean'].value_counts()

channel_clean
IVR            709
CHATBOT        698
EMAIL          375
BRANCH         366
APP            344
CALL_CENTER    308
Name: count, dtype: int64

In [41]:
# Drop original raw columns now that every field has a _clean (+ _status
# where relevant) counterpart — keeps the exported file lean for the pipeline.
raw_cols_to_drop = [
    'txn_id', 'user_id', 'merchant_id',
    'transaction_timestamp', 'reported_timestamp', 'bank_response_timestamp',
    'disputed_amount', 'reason_code', 'complaint_text',
    'resolution_status', 'severity', 'channel',
]
cb = cb.drop(columns=raw_cols_to_drop)
cb.columns.tolist()

['complaint_id',
 'txn_id_clean',
 'txn_id_status',
 'user_id_clean',
 'merchant_id_clean',
 'transaction_timestamp_clean',
 'transaction_timestamp_status',
 'reported_timestamp_clean',
 'reported_timestamp_status',
 'bank_response_timestamp_clean',
 'bank_response_timestamp_status',
 'disputed_amount_clean',
 'disputed_amount_status',
 'reason_code_clean',
 'complaint_text_clean',
 'resolution_status_clean',
 'severity_clean',
 'channel_clean']

In [42]:
cb.to_csv("../data/cleaned/modified_chargebacks.csv", index=False)